In [1]:
import pandas as pd
import numpy as np
from pathlib import Path
import os 
from thefuzz import process
import pandas as pd
import os
import re
import tkinter as tk
from tkinter import filedialog
from clase_reportes import ReportClass
rc = ReportClass()



In [2]:

# Ocultar la ventana principal de tkinter
root = tk.Tk()
root.withdraw()
root.attributes('-topmost', True)

# Paso 1: Seleccionar el archivo de ventas
nombre_archivo_ventas = filedialog.askopenfilename(
    title="Selecciona el archivo de ventas (.xlsx)",
    filetypes=[("Archivos de Excel", "*.xlsx")]
)

# Verificar si se seleccionó un archivo
if not nombre_archivo_ventas:
    print("No se seleccionó el archivo de ventas.")
    exit()

# Paso 2: Cargar el archivo de ventas
try:
    df_ventas = pd.read_excel(nombre_archivo_ventas)
    print(f"Archivo de ventas '{os.path.basename(nombre_archivo_ventas)}' cargado correctamente.")
except Exception as e:
    print(f"Error al cargar el archivo de ventas: {e}")
    exit()

# Paso 3: Seleccionar el archivo de notas crédito
nombre_archivo_notas_credito = filedialog.askopenfilename(
    title="Selecciona el archivo de notas crédito (.xlsx)",
    filetypes=[("Archivos de Excel", "*.xlsx")]
)

# Verificar si se seleccionó un archivo
if not nombre_archivo_notas_credito:
    print("No se seleccionó el archivo de notas crédito.")
    exit()

# Paso 4: Cargar el archivo de notas crédito
try:
    df_notas_credito = pd.read_excel(nombre_archivo_notas_credito)
    print(f"Archivo de notas crédito '{os.path.basename(nombre_archivo_notas_credito)}' cargado correctamente.")
except Exception as e:
    print(f"Error al cargar el archivo de notas crédito: {e}")
    exit()

df_notas_credito['NUMERO_FACTURA'] = df_notas_credito['Líneas de factura/Referencia'].apply(
    lambda x: re.search(r'(?:FEVY|FVE|FYEX|[234]YPO|YPOS|PSYA|PSYB)\d*', x).group(0) if pd.notna(x) and re.search(r'(?:FEVY|FVE|FYEX|[234]YPO|YPOS|PSYA|PSYB)\d*', x) else None
)
# Procesa las ventas para envitar fugas de información


Archivo de ventas 'ventas.xlsx' cargado correctamente.
Archivo de notas crédito 'notas_ventas.xlsx' cargado correctamente.


In [3]:
df_ventas.columns

Index(['Líneas de factura/Fecha de factura', 'Líneas de factura/Contacto',
       'Líneas de factura/Número', 'Líneas de factura/Producto',
       'Líneas de factura/Cantidad', 'Líneas de factura/Total',
       'Líneas de factura/Moneda/Tasa actual',
       'Líneas de factura/Contacto/NIF', 'Líneas de factura/Contacto/Teléfono',
       'Líneas de factura/Contacto/Correo electrónico',
       'Líneas de factura/Contacto/Ciudad',
       'Líneas de factura/Contacto/Estado', 'Equipo de ventas',
       'Líneas de factura/Referencia', 'Asesor Comercial', 'Origen',
       'Origen/Nombre de la fuente', 'Tipo de cliente', 'Etiqueta contacto',
       'Total firmado'],
      dtype='object')

In [4]:


# Guarda en la variables las ventas sin tipo de cliente y con etiqueta mayorista
# Esta variables se guarda en el archivo de errores

etiqueta_mayorista = df_ventas[(df_ventas['Tipo de cliente'].isna())&
            (df_ventas['Etiqueta contacto']=='MAYORISTA NV')
            ] 
# Copia de la etiqueta los clientes mayoristas que aparecen en blanco
df_ventas.loc[(df_ventas['Tipo de cliente'].isna())&
            (df_ventas['Etiqueta contacto']=='MAYORISTA NV'), 'Tipo de cliente'
            ] = 'MAYORISTA NV'

equipo_por_factura = (
    df_ventas
    .groupby('Líneas de factura/Número')['Equipo de ventas']
    .agg(lambda x: x.dropna().iloc[0] if not x.dropna().empty else None)
    .to_dict()
)

df_ventas['Equipo de ventas'] = df_ventas['Líneas de factura/Número'].map(equipo_por_factura)


asesor_por_factura = (
    df_ventas
    .groupby('Líneas de factura/Número')['Asesor Comercial']
    .agg(lambda x: x.dropna().iloc[0] if not x.dropna().empty else None)
    .to_dict()
)
df_ventas['Asesor Comercial'] = df_ventas['Líneas de factura/Número'].map(asesor_por_factura)

tipo_por_factura = (
    df_ventas
    .groupby('Líneas de factura/Número')['Tipo de cliente']
    .agg(lambda x: x.dropna().iloc[0] if not x.dropna().empty else None)
    .to_dict()
)
df_ventas['Tipo de cliente'] = df_ventas['Líneas de factura/Número'].map(tipo_por_factura)

######
df_notas_credito = df_notas_credito.drop(columns=['Líneas de factura/Número'])
df_notas_credito = df_notas_credito.rename(columns={'NUMERO_FACTURA': 'Líneas de factura/Número'})
# Paso 6: Convertir las cantidades y totales de las notas crédito a valores negativos
df_notas_credito['Líneas de factura/Cantidad'] = -df_notas_credito['Líneas de factura/Cantidad']
df_notas_credito['Líneas de factura/Total'] = -df_notas_credito['Líneas de factura/Total']

# Paso 8: Crear una columna temporal que combine NUMERO_FACTURA y PRODUCTO
df_ventas['NUMERO_FACTURA-PRODUCTO'] = df_ventas['Líneas de factura/Número'] + '-' + df_ventas['Líneas de factura/Producto']
df_notas_credito['NUMERO_FACTURA-PRODUCTO'] = df_notas_credito['Líneas de factura/Número'] + '-' + df_notas_credito['Líneas de factura/Producto']
# Paso 9: Filtrar las notas crédito para incluir solo las que coinciden con ventas existentes
notas_credito_validas = df_notas_credito['NUMERO_FACTURA-PRODUCTO'].isin(df_ventas['NUMERO_FACTURA-PRODUCTO'])
df_notas_credito_filtrado = df_notas_credito[notas_credito_validas]
# Paso 8: Combinar ambos datasets (ventas y notas crédito)
df_consolidado = pd.concat([df_ventas, df_notas_credito_filtrado], ignore_index=True)
# Renombrar columnas para estandarizar nombres
df_consolidado= df_consolidado.rename(columns={
    'Equipo de ventas': 'Equipo de Ventas',
    'Líneas de factura/Contacto':'Líneas de factura/Asociado',
    'Líneas de factura/Contacto/Ciudad':'Líneas de factura/Asociado/Ciudad',
    'Líneas de factura/Contacto/Correo electrónico':'Líneas de factura/Asociado/Correo electrónico',
    'Líneas de factura/Contacto/Estado':'Líneas de factura/Asociado/Estado',
    'Líneas de factura/Contacto/NIF':'Líneas de factura/Asociado/Número de Identificación',
    'Líneas de factura/Contacto/Teléfono':'Líneas de factura/Asociado/Teléfono',
    'Origen/Nombre de la fuente':'Origen/Nombre de la Fuente'

})

# Paso 10: Agrupar por la columna temporal NUMERO_FACTURA-PRODUCTO
df_consolidado = df_consolidado.groupby(
    'NUMERO_FACTURA-PRODUCTO',  # Agrupar por la combinación de factura y producto
    as_index=False
).agg({
    'Líneas de factura/Fecha de factura': 'first',
    'Líneas de factura/Asociado': 'first',
    'Líneas de factura/Número': 'first',
    'Líneas de factura/Producto': 'first',
    'Líneas de factura/Cantidad': 'sum',  # Sumar las cantidades
    'Líneas de factura/Total': 'sum',     # Sumar los totales
    'Líneas de factura/Moneda/Tasa actual': 'first',
    'Líneas de factura/Asociado/Número de Identificación': 'first',
    'Líneas de factura/Asociado/Teléfono': 'first',
    'Líneas de factura/Asociado/Correo electrónico': 'first',
    'Líneas de factura/Asociado/Ciudad': 'first',
    'Líneas de factura/Asociado/Estado': 'first',
    'Equipo de Ventas': 'first',
    'Líneas de factura/Referencia': 'first',
    'Asesor Comercial': 'first',
    'Origen': 'first',
    'Origen/Nombre de la Fuente': 'first',
    'Tipo de cliente': 'first',
    'Etiqueta contacto': 'first'

})
# Paso 12: Eliminar la columna temporal NUMERO_FACTURA-PRODUCTO
df_consolidado.drop(columns=['NUMERO_FACTURA-PRODUCTO'], inplace=True)
# Paso 13: Filtrar solo las filas donde la cantidad sea mayor que 0 (eliminar ventas canceladas)
df_consolidado = df_consolidado[df_consolidado['Líneas de factura/Cantidad'] > 0]
# Paso 14: Generar el nombre del archivo de salida
nombre_archivo= os.path.splitext(os.path.split(nombre_archivo_ventas)[-1])[0]

# Paso 15: Guardar el archivo consolidado
try:
    ruta = rc.validar_ruta()
    ruta_salida = ruta / 'RAW DATA' / 'PROCESADO' / f'{nombre_archivo}_procesado.xlsx'
    if not ruta_salida.parent.exists():
        ruta_salida.parent.mkdir(parents=True, exist_ok=True)
    df_consolidado.to_excel(ruta_salida, index=False)
    print(f"Archivo consolidado guardado como '{ruta_salida}'.")
except Exception as e:
    print(f"Error al guardar el archivo consolidado: {e}")

# Paso 16: Obtener el listado de facturas afectadas por notas crédito
facturas_afectadas = df_notas_credito_filtrado[['Líneas de factura/Número', 'Líneas de factura/Producto', 'Líneas de factura/Cantidad', 'Líneas de factura/Total']].dropna(subset=['Líneas de factura/Número'])

# Paso 17: Mostrar el listado de facturas afectadas
print("Facturas afectadas por notas crédito:")
print(facturas_afectadas.shape)


try:
    nombre_archivo_facturas_afectadas = ruta / 'RAW DATA' / 'FACTURAS AFECTADAS' / f'{nombre_archivo}_facturas_afectadas.xlsx' 
    facturas_afectadas.to_excel(nombre_archivo_facturas_afectadas, index=False)
    print(f"Listado de facturas afectadas guardado como '{nombre_archivo_facturas_afectadas}'.")
except Exception as e:
    print(f"Error al guardar el listado de facturas afectadas: {e}")

# return {'archivo_salida': df_consolidado,
#         'nombre_archivo':ruta_salida,
#         'facturas_afectadas' :facturas_afectadas}


Archivo consolidado guardado como 'C:\Users\Dataa\Desktop\VENTAS\VENTA MENSUAL\RAW DATA\PROCESADO\ventas_procesado.xlsx'.
Facturas afectadas por notas crédito:
(2868, 4)
Listado de facturas afectadas guardado como 'C:\Users\Dataa\Desktop\VENTAS\VENTA MENSUAL\RAW DATA\FACTURAS AFECTADAS\ventas_facturas_afectadas.xlsx'.


In [5]:

@staticmethod
def limpiar_documento(valor):
    """
    Esta función limpia y extrae el número de identificación de un valor dado.
    Args:
        valor (str/int/float): Valor que contiene el número de identificación.
    Returns:
        str/pd.NA: Número de identificación limpio o pd.NA si no es válido.
    """
    valor = str(valor).strip()
    valor = valor.replace('.', '').replace("'",'').replace("”", '').replace("’",'')  # Elimina espacios en blanco

    # Si el valor es NaN o vacío, retorna NaN
    if pd.isna(valor) or str(valor).strip() == pd.NaT:
        return pd.NA
    # Si el valor contiene solo letras, retorna NaN
    if re.fullmatch(r'[A-Za-z]+', str(valor).strip()):
        return pd.NA
    # Extrae los dígitos iniciales antes de cualquier carácter no numérico
    match = re.match(r'^(\d+)', str(valor))
    if match:
        return match.group(1)
    return pd.NA    




In [6]:
# origen = True


# if origen:
#     notas_creditos = rc.notas_creditos()
#     nombre_archivo = notas_creditos['nombre_archivo']
#     df = notas_creditos['archivo_salida']
# else:
#     # Ocultar la ventana principal de tkinter
#     root = tk.Tk()
#     root.withdraw()
#     root.attributes('-topmost', True)
#     # Paso 1: Seleccionar el archivo de ventas
#     nombre_archivo_ventas = filedialog.askopenfilename(
#         title="Por favor, ingresa el nombre del archivo de ventas (incluye la extensión .xlsx): ",
#         filetypes=[("Archivos de Excel", "*.xlsx")]
#     )

#     # Verificar si se seleccionó un archivo
#     if not nombre_archivo_ventas:
#         print("No se seleccionó el archivo.")
#         exit()

#     # Paso 2: Cargar el archivo de ventas
#     try:
#         df = pd.read_excel(nombre_archivo_ventas)
#         nombre_archivo = os.path.basename(nombre_archivo_ventas)
#         print(f"Archivo  '{nombre_archivo}' cargado correctamente.")
#     except Exception as e:
#         print(f"Error al cargar el archivo: {e}")
#         exit()


df = df_consolidado.copy()

# Extraer el código de país y reemplazar NaN con "Desconocido" en un solo paso
df['pais'] = df['Líneas de factura/Asociado/Estado'].str.extract(r'\(([A-Z]{2})\)').fillna("Desconocido")

# Crear la columna 'total' con la lógica especificada
df['Líneas de factura/Asociado/Ciudad'] = df['Líneas de factura/Asociado/Ciudad'].astype(str).fillna("Desconocido")

df_filtrado = df.copy()

# # La idea es subir esta logica########


# # Guarda en la variables las ventas sin tipo de cliente y con etiqueta mayorista
# # Esta variables se guarda en el archivo de errores
# etiqueta_mayorista = df_filtrado[(df_filtrado['Tipo de cliente'].isna())&
#             (df_filtrado['Etiqueta contacto']=='MAYORISTA NV')
#             ] 
# # Copia de la etiqueta los clientes mayoristas que aparecen en blanco
# df_filtrado.loc[(df_filtrado['Tipo de cliente'].isna())&
#             (df_filtrado['Etiqueta contacto']=='MAYORISTA NV'), 'Tipo de cliente'
#             ] = 'MAYORISTA NV'

# equipo_por_factura = (
#     df_filtrado
#     .groupby('Líneas de factura/Número')['Equipo de Ventas']
#     .agg(lambda x: x.dropna().iloc[0] if not x.dropna().empty else None)
#     .to_dict()
# )

# df_filtrado['Equipo de Ventas'] = df_filtrado['Líneas de factura/Número'].map(equipo_por_factura)

# ########

# asesor_por_factura = (
#     df_filtrado
#     .groupby('Líneas de factura/Número')['Asesor Comercial']
#     .agg(lambda x: x.dropna().iloc[0] if not x.dropna().empty else None)
#     .to_dict()
# )
# df_filtrado['Asesor Comercial'] = df_filtrado['Líneas de factura/Número'].map(asesor_por_factura)

# tipo_por_factura = (
#     df_filtrado
#     .groupby('Líneas de factura/Número')['Tipo de cliente']
#     .agg(lambda x: x.dropna().iloc[0] if not x.dropna().empty else None)
#     .to_dict()
# )
# df_filtrado['Tipo de cliente'] = df_filtrado['Líneas de factura/Número'].map(tipo_por_factura)


df_filtrado = df_filtrado[df_filtrado['Líneas de factura/Producto'].str.startswith(('[PCN','[KD','[TNG','[B8'))].copy()  

#######



In [7]:

# Ahora puedes modificar df_filtrado sin preocuparte por el warning
df_filtrado.loc[:, 'Líneas de factura/Fecha de factura'] = pd.to_datetime(df_filtrado['Líneas de factura/Fecha de factura'])
# df_filtrado['Líneas de factura/Fecha de factura'] = pd.to_datetime(df_filtrado['Líneas de factura/Fecha de factura'])

df_filtrado = df_filtrado.reset_index(drop=True)
# Convertir la columna 'Líneas de factura/Total' a tipo numérico
df_filtrado['Líneas de factura/Total'] = pd.to_numeric(df_filtrado['Líneas de factura/Total'], errors='coerce')
# df_filtrado.loc['', 'Líneas de factura/Total'] = pd.to_numeric(df_filtrado['Líneas de factura/Total'], errors='coerce')
# Verificar si hay valores nulos después de la conversión
print("Valores nulos en 'Líneas de factura/Total':", df_filtrado['Líneas de factura/Total'].isnull().sum())
# Paso 1: Verificar valores nulos en la columna de fecha
print("Valores nulos en 'Líneas de factura/Fecha de factura':", df_filtrado['Líneas de factura/Fecha de factura'].isnull().sum())
df_filtrado = df_filtrado.dropna(subset=['Líneas de factura/Fecha de factura'])


#  Leer el CSV desde la URL
url = "https://www.datos.gov.co/resource/32sa-8pi3.csv"
df_TRM = pd.read_csv(url)

#  Cambiar tipos de datos
df_TRM['valor'] = pd.to_numeric(df_TRM['valor'], errors='coerce')
df_TRM['unidad'] = df_TRM['unidad'].astype(str)
df_TRM['vigenciadesde'] = pd.to_datetime(df_TRM['vigenciadesde'], errors='coerce')
df_TRM['vigenciahasta'] = pd.to_datetime(df_TRM['vigenciahasta'], errors='coerce')

#  Crear una nueva columna con el año de 'vigenciadesde'
df_TRM['Año'] = df_TRM['vigenciadesde'].dt.year

#  Filtrar por el año 2025
today = pd.to_datetime('now')
df_TRM = df_TRM[df_TRM['Año'] == today.year]
df_TRM['TRM'] = df_TRM['valor']

# Crear una lista para almacenar las filas expandidas
expanded_rows = []

# Iterar sobre cada fila de df_TRM y generar las fechas dentro del rango de vigencia
for _, row in df_TRM.iterrows():
    date_range = pd.date_range(start=row['vigenciadesde'], end=row['vigenciahasta'], freq='D')
    for date in date_range:
        expanded_rows.append({'Fecha': date, 'TRM': row['TRM']})

# Crear un nuevo DataFrame a partir de la lista
df_TRM_expandido = pd.DataFrame(expanded_rows)

# Eliminar duplicados (si los hay)
df_TRM_expandido = df_TRM_expandido.drop_duplicates(subset=['Fecha'])

# Ordenar por fecha
df_TRM_expandido = df_TRM_expandido.sort_values('Fecha')

# Verificar el nuevo DataFrame


df_filtrado['Líneas de factura/Fecha de factura'] = pd.to_datetime(df_filtrado['Líneas de factura/Fecha de factura'])
df_TRM_expandido['Fecha'] = pd.to_datetime(df_TRM_expandido['Fecha'])
df_filtrado = df_filtrado.sort_values(by='Líneas de factura/Fecha de factura')
df_TRM_expandido = df_TRM_expandido.sort_values(by='Fecha')

# Intentar el merge_asof
try:
    df_resultado = pd.merge_asof(
        df_filtrado,
        df_TRM_expandido[['Fecha', 'TRM']],  # Mantener solo las columnas necesarias
        left_on='Líneas de factura/Fecha de factura',
        right_on='Fecha',
        direction='backward'  # Tomar la TRM vigente más reciente anterior o igual a la fecha
    )
    
    # Verificar si la columna TRM está vacía
    if df_resultado['TRM'].isnull().all():
        print("Advertencia: La columna TRM está vacía después del merge. Verifica las fechas y los datos.")
    else:
        print("Merge completado correctamente.")
except Exception as e:
    print(f"Error al realizar el merge: {str(e)}")
    import traceback
    traceback.print_exc()

df_resultado['total'] = df_resultado.apply(
    lambda row: row['Líneas de factura/Total'] if row['pais'] in ['CO', 'Desconocido'] else row['Líneas de factura/Total'] * row['TRM'],
    axis=1
)


ciudad_url = "https://www.datos.gov.co/resource/gdxc-w37w.csv?$limit=5000"
DF_CIUDADES = pd.read_csv(ciudad_url)
# DF_CIUDADES = pd.read_excel(r"C:\Users\Dataa\Desktop\VENTAS\VENTA MENSUAL\CIUDAD.xlsx") # Dataset con nombres correctos
DF_CIUDADES = DF_CIUDADES.rename(columns= {'nom_mpio':'Ciudad_Correcta'})
df_resultado = df_resultado.rename(columns= {'Líneas de factura/Asociado/Ciudad':'Ciudad'})

# 2️⃣ Definir los nombres de columnas
col_ciudad_correcta = "Ciudad_Correcta"  # Nombre en DF_CIUDADES
col_ciudad_ventas = "Ciudad"  # Nombre en DF_VENTAS

# 3️⃣ Convertir todas las ciudades a string y manejar NaN
df_resultado[col_ciudad_ventas] = df_resultado[col_ciudad_ventas].astype(str).fillna("Desconocido")


# 4️⃣ Lista de ciudades correctas (convertidas a string)
lista_ciudades_correctas = DF_CIUDADES[col_ciudad_correcta].astype(str).unique()

# 5️⃣ Función para encontrar la mejor coincidencia
def corregir_ciudad(ciudad_mal):
    if ciudad_mal.lower() == "nan" or ciudad_mal.strip() == "":
        return "Desconocido"  # Manejar valores vacíos o NaN
    mejor_match, score = process.extractOne(ciudad_mal, lista_ciudades_correctas)
    return mejor_match if score >= 60 else ciudad_mal  # Si el match es bajo, dejar el original

# 5️⃣ Aplicar la función a la columna de ciudades en ventas
df_resultado["Ciudad_Corregida"] = df_resultado[col_ciudad_ventas].apply(corregir_ciudad)



# 6️⃣ Convertir la columna "Ciudad_Corregida" a mayúsculas
df_resultado["Ciudad_Corregida"] = df_resultado["Ciudad_Corregida"].str.upper()

# Diccionario para renombrar las columnas
nuevos_nombres = {
    'Líneas de factura/Fecha de factura': 'Fecha_Factura',
    'Líneas de factura/Asociado': 'Cliente',
    'Líneas de factura/Número': 'Numero_Factura',
    'Líneas de factura/Producto': 'Producto',
    'Líneas de factura/Cantidad': 'Cantidad',
    'Líneas de factura/Total': 'Total',
    'Líneas de factura/Moneda/Tasa actual': 'Tasa_Cambio',
    'Líneas de factura/Asociado/Número de Identificación': 'Identificacion_Cliente',
    'Líneas de factura/Asociado/Teléfono': 'Telefono',
    'Líneas de factura/Asociado/Correo electrónico': 'Email',
    'Ciudad': 'Ciudad',
    'Líneas de factura/Asociado/Estado': 'Departamento',
    'Equipo de Ventas': 'Equipo_Ventas',
    'Líneas de factura/Referencia': 'Referencia',
    'pais': 'Pais',
    'Fecha': 'Fecha_TRM',
    'TRM': 'TRM',
    'total': 'Total($)',
    'Ciudad_Corregida': 'Ciudad_Corregida'
}

# Renombrar las columnas
df_resultado = df_resultado.rename(columns=nuevos_nombres)

# Verificar el resultado

# Extraer el día, mes y año en nuevas columnas
df_resultado['Dia'] = df_resultado['Fecha_Factura'].dt.day
df_resultado['Mes'] = df_resultado['Fecha_Factura'].dt.month
df_resultado['Año'] = df_resultado['Fecha_Factura'].dt.year


# Reorganizar las columnas si es necesario
column_order = ['Numero_Factura','Fecha_Factura', 'Dia', 'Mes', 'Año', 'Cliente','Identificacion_Cliente','Producto', 'Cantidad', 
                'Total', 'Tasa_Cambio','TRM', 'Total($)','Telefono', 'Email','Pais','Ciudad', 'Ciudad_Corregida', 'Departamento', 
                'Equipo_Ventas', 'Referencia', 'Asesor Comercial', 'Tipo de cliente'] ## aqui se agregregan las nuevas columnas
df_resultado = df_resultado[column_order]



# Convertir la columna "Cliente" a mayúsculas
df_resultado['Cliente'] = df_resultado['Cliente'].str.upper()

# Eliminar espacios en blanco al principio y al final de cada valor en la columna "Cliente"
df_resultado['Cliente'] = df_resultado['Cliente'].str.strip()
# Convertir la columna "Producto" a mayúsculas
df_resultado['Producto'] = df_resultado['Producto'].str.upper()
# Eliminar espacios en blanco al principio y al final de cada valor en la columna "Producto"
df_resultado['Producto'] = df_resultado['Producto'].str.strip()
# Convertir los nombres de las columnas a mayúsculas
df_resultado.columns = df_resultado.columns.str.upper()


# 2. Limpieza de la columna de identificación en ambos DataFrames
# Limpieza en tu DataFrame actual
df_resultado['IDENTIFICACION_CLIENTE'] = (
    df_resultado['IDENTIFICACION_CLIENTE']
    .astype(str)  # Convertir a string
    .str.strip()  # Eliminar espacios al principio y al final
    .str.replace(r'\s+', '', regex=True)  # Eliminar espacios adicionales entre caracteres
)


columnas = df_resultado.columns.tolist()

# Encontramos la posición de la columna "Producto"
posicion_producto = columnas.index('PRODUCTO')

# Movemos "Categoría" antes de "Producto"
columnas.insert(posicion_producto, columnas.pop(columnas.index('TIPO DE CLIENTE')))

# Reorganizamos el DataFrame
df_resultado = df_resultado[columnas]

# Rellenar los valores NaN en "Categoría" cuando EQUIPO_VENTAS sea "Shopify"
df_resultado.loc[df_resultado['EQUIPO_VENTAS'] == 'Shopify', 'TIPO DE CLIENTE'] = 'SHOPIFY'
# Rellenar los valores NaN en "Categoría" cuando EQUIPO_VENTAS sea "Shopify"
df_resultado.loc[df_resultado['EQUIPO_VENTAS'] == 'Punto de venta', 'TIPO DE CLIENTE'] = 'CALL CENTER'

cliente_call_center = df_resultado[df_resultado['TIPO DE CLIENTE']=='CLIENTE']
# Reemplazar "CLIENTE" por "CALL CENTER" en la columna "CATEGORÍA"
df_resultado.loc[df_resultado['TIPO DE CLIENTE']=='CLIENTE', 'TIPO DE CLIENTE'] = 'CALL CENTER'
# df_resultado[df_resultado['CATEGORÍA'].isna()].to_excel(r"C:\Users\Dataa\Desktop\ventas_sin_categoria.xlsx")

df_resultado.loc[~df_resultado['PAIS'].isin(['CO', 'Desconocido']), 'TIPO DE CLIENTE'] = df_resultado['PAIS']


# Rellenar los valores vacíos en "Categoría" con "Call center"
df_resultado['TIPO DE CLIENTE'] =df_resultado['TIPO DE CLIENTE'].fillna('CALL CENTER')   ### REVISAR


df_resultado= df_resultado.rename(columns={'TIPO DE CLIENTE':'CATEGORÍA'})


# Renombra las categorías

categorias_renombrar = {
    'Catalogo': 'CATÁLOGO',
    'Distribuidor': 'DISTRIBUIDOR',
    'Empleado': 'EMPLEADO',
    'FARMACIAS': 'FARMACIA',
    'HOLE COSMETICS':'HOLE COSMETICS SAS',
    'Surticosmeticos': 'SURTICOSMETICOS'
}

df_resultado['CATEGORÍA'] = df_resultado['CATEGORÍA'].replace(categorias_renombrar)

# validar clientes nuevos, pendiente por terminar
try:
    ruta = rc.validar_ruta()
    ruta_historoica = ruta / 'file' / f'ventas_historicas.xlsx'

    

    ventas_historicas_agru=pd.read_excel(ruta_historoica)
    ventas_historicas_agru['FECHA_FACTURA'] = pd.to_datetime(ventas_historicas_agru['FECHA_FACTURA'])

    ventas_historicas_agru['IDENTIFICACION_CLIENTE'] =  ventas_historicas_agru['IDENTIFICACION_CLIENTE'].apply(self.limpiar_documento)
    ventas_historicas_agru['IDENTIFICACION_CLIENTE'].astype(str)



    df_resultado = df_resultado.merge(ventas_historicas_agru, on='IDENTIFICACION_CLIENTE', how='left', suffixes=('', '_FECHA_MIN'))

    now = pd.to_datetime('now')

    df_resultado['CLIENTES NUEVOS'] = np.where(
        ((df_resultado['FECHA_FACTURA_FECHA_MIN'].dt.month == now.month) & 
        (df_resultado['FECHA_FACTURA_FECHA_MIN'].dt.year == now.year))|((df_resultado['FECHA_FACTURA_FECHA_MIN'].isna())&(df_resultado['CATEGORÍA']=='MAYORISTA NV')),
        "Cliente sin historico",
        ""
    )

    df_resultado = df_resultado.drop(columns=['FECHA_FACTURA_FECHA_MIN'])
except Exception as e:
    print(f"Error al validar clientes nuevos: {e}")

# return  {'Base':df_resultado,
#         'nombre_archivo':notas_creditos['nombre_archivo'],
#         'facturas_afectadas':notas_creditos['facturas_afectadas'],
#         'errores':etiqueta_mayorista,
#         # 'asesores_sin_categoria':asesores_sin_categoria,
#         'cliente_call_center':cliente_call_center
#             }


Valores nulos en 'Líneas de factura/Total': 0
Valores nulos en 'Líneas de factura/Fecha de factura': 0
Merge completado correctamente.
Error al validar clientes nuevos: name 'self' is not defined
